In [0]:
%pip install -q transformers torch sentencepiece accelerate
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

import re
import time
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

# ============================================================
# CONFIG
# ============================================================
OCR_TABLE   = "noc_documents_multilingual"
TEXT_COL    = "final_text"
EMBED_COL   = "embedding"

KEY_COL     = None  

MIN_TOKENS  = 800
MAX_TOKENS  = 1200
OVERLAP_PCT = 0.15
E5_PREFIX   = "passage: "

BATCH_ROWS      = 500     
NUM_PARTITIONS  = 2       
MAX_LEN         = 512
BATCH_SIZE      = 1        

MAX_RETRIES     = 3
SLEEP_BASE_SEC  = 10

STAGING_TABLE   = f"{OCR_TABLE}__e5_staging"  

spark.conf.set("spark.sql.shuffle.partitions", str(max(4, NUM_PARTITIONS)))

# ============================================================
# LOAD TABLE + KEY
# ============================================================
base_df = spark.table(f"`{OCR_TABLE}`")

if KEY_COL is None:
    cols = [c.lower() for c in base_df.columns]
    if "path" in cols:
        KEY_COL = base_df.columns[cols.index("path")]
    elif "id" in cols:
        KEY_COL = base_df.columns[cols.index("id")]
    else:
        raise Exception("Set KEY_COL to a unique identifier column (path or id).")

print("Using KEY_COL =", KEY_COL)
key_dtype = base_df.schema[KEY_COL].dataType

# ============================================================
# TEXT CLEANING + CHUNKING
# ============================================================
_MODEL = None
_TOKENIZER = None
_DEVICE = None

_CTRL_CHARS_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")
_WS_RE = re.compile(r"\s+")
_SENT_SPLIT_RE = re.compile(r"(?<=[\.\!\?\;\:])\s+|\n+")

def _get_model():
    global _MODEL, _TOKENIZER, _DEVICE
    if _MODEL is None or _TOKENIZER is None:
        model_name = "intfloat/multilingual-e5-large"
        _TOKENIZER = AutoTokenizer.from_pretrained(model_name)
        _MODEL = AutoModel.from_pretrained(model_name)
        _MODEL.eval()
        _DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        _MODEL.to(_DEVICE)
    return _MODEL, _TOKENIZER, _DEVICE

def _clean_text(s: str) -> str:
    if s is None:
        return ""
    s = _CTRL_CHARS_RE.sub(" ", s)
    s = _WS_RE.sub(" ", s).strip()
    return s

def _sentences(s: str):
    parts = _SENT_SPLIT_RE.split(s)
    return [p.strip() for p in parts if p and p.strip()]

def _build_chunks(sent_list, tokenizer, min_tokens, max_tokens, overlap_pct):
    if not sent_list:
        return []

    overlap_tokens = max(1, int(max_tokens * overlap_pct))
    chunks = []
    cur_sents = []
    cur_tok = 0

    sent_tok_lens = [len(tokenizer.encode(E5_PREFIX + t, add_special_tokens=False)) for t in sent_list]

    i = 0
    while i < len(sent_list):
        s = sent_list[i]
        s_tok = sent_tok_lens[i]

        if s_tok > max_tokens:
            if cur_sents:
                chunks.append(" ".join(cur_sents))
                cur_sents, cur_tok = [], 0

            ids = tokenizer.encode(E5_PREFIX + s, add_special_tokens=False)
            start = 0
            while start < len(ids):
                end = min(start + max_tokens, len(ids))
                sub = tokenizer.decode(ids[start:end])
                chunks.append(sub)
                if end == len(ids):
                    break
                start = max(0, end - overlap_tokens)
            i += 1
            continue

        if cur_tok + s_tok <= max_tokens:
            cur_sents.append(s)
            cur_tok += s_tok
            i += 1
        else:
            if cur_sents:
                chunks.append(" ".join(cur_sents))
            else:
                chunks.append(s)
                i += 1

            overlap_sents = []
            overlap_count = 0
            j = len(cur_sents) - 1
            while j >= 0 and overlap_count < overlap_tokens:
                overlap_sents.append(cur_sents[j])
                overlap_count += len(tokenizer.encode(E5_PREFIX + cur_sents[j], add_special_tokens=False))
                j -= 1
            overlap_sents.reverse()

            cur_sents = overlap_sents
            cur_tok = 0
            for os in cur_sents:
                cur_tok += len(tokenizer.encode(E5_PREFIX + os, add_special_tokens=False))

    if cur_sents:
        chunks.append(" ".join(cur_sents))

    return chunks

# ============================================================
# EMBEDDING
# ============================================================
@torch.no_grad()
def _embed_texts(texts, model, tokenizer, device):
    all_emb = []
    for b0 in range(0, len(texts), BATCH_SIZE):
        batch = texts[b0:b0+BATCH_SIZE]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        out = model(**enc)
        last = out.last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1)

        summed = (last * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1)
        emb = summed / counts
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)

        all_emb.append(emb.detach().cpu().numpy())

        del enc, out, last, mask, summed, counts, emb
        if device.type == "cuda":
            torch.cuda.empty_cache()

    return np.vstack(all_emb)

def _embed_document(doc_text: str):
    model, tokenizer, device = _get_model()

    cleaned = _clean_text(doc_text)
    if not cleaned:
        return None

    sents = _sentences(cleaned)
    chunks = _build_chunks(sents, tokenizer, MIN_TOKENS, MAX_TOKENS, OVERLAP_PCT)
    if not chunks:
        return None

    chunk_inputs = [E5_PREFIX + c for c in chunks]
    chunk_emb = _embed_texts(chunk_inputs, model, tokenizer, device)

    weights = [len(tokenizer.encode(E5_PREFIX + c, add_special_tokens=False)) for c in chunks]
    w = np.array(weights, dtype=np.float32)
    w = w / max(w.sum(), 1.0)

    doc_emb = (chunk_emb * w.reshape(-1, 1)).sum(axis=0)
    norm = np.linalg.norm(doc_emb)
    if norm > 0:
        doc_emb = doc_emb / norm

    return doc_emb.astype(np.float32).tolist()

# ============================================================
# MAP IN PANDAS
# ============================================================
out_schema = T.StructType([
    T.StructField(KEY_COL, key_dtype, nullable=False),
    T.StructField(EMBED_COL, T.ArrayType(T.FloatType()), nullable=True),
    T.StructField("embed_error", T.StringType(), nullable=True),
])

def embed_map(pdf_iter):
    for pdf in pdf_iter:
        keys = pdf[KEY_COL].tolist()
        texts = pdf[TEXT_COL].astype(str).tolist()

        embs = []
        errs = []
        for t in texts:
            try:
                v = _embed_document(t)
                embs.append(v)
                errs.append(None if v is not None else "embed_document_returned_None")
            except Exception as e:
                embs.append(None)
                errs.append(str(e)[:2000])

        yield pd.DataFrame({KEY_COL: keys, EMBED_COL: embs, "embed_error": errs})

def progress():
    spark.table(f"`{OCR_TABLE}`").selectExpr(
        "count(*) as total_rows",
        f"sum(case when {EMBED_COL} is not null then 1 else 0 end) as completed_embeddings",
        f"sum(case when {EMBED_COL} is null then 1 else 0 end) as remaining_embeddings",
        f"round(100.0 * sum(case when {EMBED_COL} is not null then 1 else 0 end)/count(*), 4) as pct_complete"
    ).show(truncate=False)

# ============================================================
# STAGING TABLE 
# ============================================================
spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{STAGING_TABLE}` (
  `{KEY_COL}` {key_dtype.simpleString()},
  `{EMBED_COL}` ARRAY<FLOAT>,
  `embed_error` STRING,
  `batch_id` BIGINT,
  `ingested_at` TIMESTAMP
)
USING DELTA
""")

# ============================================================
# BATCH ORCHESTRATOR 
# ============================================================
def next_batch_df(limit_rows: int):
    return (
        spark.table(f"`{OCR_TABLE}`")
             .where(F.col(TEXT_COL).isNotNull() & (F.length(F.trim(F.col(TEXT_COL))) > 0))
             .where(F.col(EMBED_COL).isNull())
             .select(F.col(KEY_COL), F.col(TEXT_COL))
             .orderBy(F.col(KEY_COL))
             .limit(limit_rows)
    )

batch_id = 0
progress()

while True:
    batch_df = next_batch_df(BATCH_ROWS)

    if len(batch_df.limit(1).collect()) == 0:
        print("✅ Done: no rows left with embedding IS NULL.")
        break

    batch_id += 1
    print(f"\n====================")
    print(f"Starting batch_id={batch_id}")
    print(f"====================")

    attempt = 0
    while True:
        attempt += 1
        try:
            emb_df = (
                batch_df
                .repartition(NUM_PARTITIONS, F.col(KEY_COL))
                .mapInPandas(embed_map, schema=out_schema)
                .withColumn("batch_id", F.lit(batch_id).cast("bigint"))
                .withColumn("ingested_at", F.current_timestamp())
            )

            (emb_df
             .write
             .format("delta")
             .mode("append")
             .saveAsTable(STAGING_TABLE))

            print(f"✅ Batch {batch_id} embedded and appended to staging table `{STAGING_TABLE}`")

            spark.sql(f"""
            MERGE INTO `{OCR_TABLE}` AS t
            USING (
              SELECT `{KEY_COL}` as `{KEY_COL}`, `{EMBED_COL}` as `{EMBED_COL}`
              FROM `{STAGING_TABLE}`
              WHERE batch_id = {batch_id} AND `{EMBED_COL}` IS NOT NULL
            ) AS s
            ON t.`{KEY_COL}` = s.`{KEY_COL}`
            WHEN MATCHED AND t.`{EMBED_COL}` IS NULL
              THEN UPDATE SET t.`{EMBED_COL}` = s.`{EMBED_COL}`
            """)

            print(f"✅ Batch {batch_id} merge complete.")
            progress()
            break

        except Exception as e:
            print(f"❌ Batch {batch_id} attempt {attempt} failed: {str(e)[:1200]}")
            if attempt >= MAX_RETRIES:
                print("Stopping after max retries. Your progress so far is محفوظ in the staging table.")
                raise
            sleep_s = SLEEP_BASE_SEC * attempt
            print(f"Retrying batch {batch_id} in {sleep_s} seconds ...")
            time.sleep(sleep_s)

print("\nAll batches finished.")

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

import re
import os
import time
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

# -----------------------------
# CONFIG
# -----------------------------
OCR_TABLE   = "noc_documents_multilingual"
TEXT_COL    = "final_text"
EMBED_COL   = "embedding"
KEY_COL     = None  

MIN_TOKENS  = 800
MAX_TOKENS  = 1200
OVERLAP_PCT = 0.15
E5_PREFIX   = "passage: "


BATCH_ROWS      = 500      
NUM_PARTITIONS  = 4     

MAX_LEN         = 512
BATCH_SIZE      = 8       

MAX_RETRIES     = 3
SLEEP_BASE_SEC  = 10


DBFS_MODEL_DIR  = "/dbfs/FileStore/models/intfloat_multilingual_e5_large"
HF_CACHE_DIR    = "/dbfs/FileStore/hf_cache" 

STAGING_TABLE   = f"{OCR_TABLE}__e5_staging"


spark.conf.set("spark.sql.shuffle.partitions", str(max(8, NUM_PARTITIONS * 2)))

# -----------------------------
# LOG HELPERS
# -----------------------------
def log(msg: str):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

def progress(label="progress"):
    df = spark.table(f"`{OCR_TABLE}`").selectExpr(
        "count(*) as total_rows",
        f"sum(case when {EMBED_COL} is not null then 1 else 0 end) as completed_embeddings",
        f"sum(case when {EMBED_COL} is null then 1 else 0 end) as remaining_embeddings",
        f"round(100.0 * sum(case when {EMBED_COL} is not null then 1 else 0 end)/count(*), 4) as pct_complete"
    )
    log(f"{label}:")
    df.show(truncate=False)

# -----------------------------
# LOAD TABLE + KEY
# -----------------------------
base_df = spark.table(f"`{OCR_TABLE}`")

if KEY_COL is None:
    cols = [c.lower() for c in base_df.columns]
    if "path" in cols:
        KEY_COL = base_df.columns[cols.index("path")]
    elif "id" in cols:
        KEY_COL = base_df.columns[cols.index("id")]
    else:
        raise Exception("Set KEY_COL to a unique identifier column (path or id).")

log(f"Using KEY_COL = {KEY_COL}")
key_dtype = base_df.schema[KEY_COL].dataType

# -----------------------------
# ENSURE MODEL EXISTS ON DBFS (DRIVER)
# -----------------------------
def ensure_model_on_dbfs():
    os.environ["HF_HOME"] = HF_CACHE_DIR  # optional shared cache location
    if os.path.exists(DBFS_MODEL_DIR) and len(os.listdir(DBFS_MODEL_DIR)) > 0:
        log(f"Model already present on DBFS: {DBFS_MODEL_DIR}")
        return

    log("Model not found on DBFS. Downloading from HuggingFace Hub to DBFS (driver)...")
    t0 = time.time()

    try:
        from huggingface_hub import snapshot_download
    except Exception as e:
        raise Exception(
            "huggingface_hub not installed. Install it with: %pip install huggingface_hub"
        ) from e

    snapshot_download(
        repo_id="intfloat/multilingual-e5-large",
        local_dir=DBFS_MODEL_DIR,
        local_dir_use_symlinks=False
    )

    log(f"Downloaded model to {DBFS_MODEL_DIR} in {round(time.time() - t0, 2)}s")

ensure_model_on_dbfs()

# -----------------------------
# TEXT CLEANING + CHUNKING
# -----------------------------
_MODEL = None
_TOKENIZER = None
_DEVICE = None

_CTRL_CHARS_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")
_WS_RE = re.compile(r"\s+")
_SENT_SPLIT_RE = re.compile(r"(?<=[\.\!\?\;\:])\s+|\n+")

def _get_model():
    """Executor-safe: loads from DBFS local path only."""
    global _MODEL, _TOKENIZER, _DEVICE
    if _MODEL is None or _TOKENIZER is None:
        model_path = DBFS_MODEL_DIR  # "/dbfs/..." path is accessible on workers too
        _TOKENIZER = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
        _MODEL = AutoModel.from_pretrained(model_path, local_files_only=True)
        _MODEL.eval()
        _DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        _MODEL.to(_DEVICE)
    return _MODEL, _TOKENIZER, _DEVICE

def _clean_text(s: str) -> str:
    if s is None:
        return ""
    s = _CTRL_CHARS_RE.sub(" ", s)
    s = _WS_RE.sub(" ", s).strip()
    return s

def _sentences(s: str):
    parts = _SENT_SPLIT_RE.split(s)
    return [p.strip() for p in parts if p and p.strip()]

def _build_chunks(sent_list, tokenizer, min_tokens, max_tokens, overlap_pct):
    if not sent_list:
        return []

    overlap_tokens = max(1, int(max_tokens * overlap_pct))
    chunks = []
    cur_sents = []
    cur_tok = 0

    sent_tok_lens = [len(tokenizer.encode(E5_PREFIX + t, add_special_tokens=False)) for t in sent_list]

    i = 0
    while i < len(sent_list):
        s = sent_list[i]
        s_tok = sent_tok_lens[i]

        if s_tok > max_tokens:
            if cur_sents:
                chunks.append(" ".join(cur_sents))
                cur_sents, cur_tok = [], 0

            ids = tokenizer.encode(E5_PREFIX + s, add_special_tokens=False)
            start = 0
            while start < len(ids):
                end = min(start + max_tokens, len(ids))
                sub = tokenizer.decode(ids[start:end])
                chunks.append(sub)
                if end == len(ids):
                    break
                start = max(0, end - overlap_tokens)
            i += 1
            continue

        if cur_tok + s_tok <= max_tokens:
            cur_sents.append(s)
            cur_tok += s_tok
            i += 1
        else:
            if cur_sents:
                chunks.append(" ".join(cur_sents))
            else:
                chunks.append(s)
                i += 1

            overlap_sents = []
            overlap_count = 0
            j = len(cur_sents) - 1
            while j >= 0 and overlap_count < overlap_tokens:
                overlap_sents.append(cur_sents[j])
                overlap_count += len(tokenizer.encode(E5_PREFIX + cur_sents[j], add_special_tokens=False))
                j -= 1
            overlap_sents.reverse()

            cur_sents = overlap_sents
            cur_tok = 0
            for osent in cur_sents:
                cur_tok += len(tokenizer.encode(E5_PREFIX + osent, add_special_tokens=False))

    if cur_sents:
        chunks.append(" ".join(cur_sents))

    return chunks

# -----------------------------
# EMBEDDING
# -----------------------------
@torch.no_grad()
def _embed_texts(texts, model, tokenizer, device):
    all_emb = []
    for b0 in range(0, len(texts), BATCH_SIZE):
        batch = texts[b0:b0+BATCH_SIZE]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        out = model(**enc)
        last = out.last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1)

        summed = (last * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1)
        emb = summed / counts
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)

        all_emb.append(emb.detach().cpu().numpy())

        del enc, out, last, mask, summed, counts, emb

    return np.vstack(all_emb)

def _embed_document(doc_text: str):
    model, tokenizer, device = _get_model()

    cleaned = _clean_text(doc_text)
    if not cleaned:
        return None

    sents = _sentences(cleaned)
    chunks = _build_chunks(sents, tokenizer, MIN_TOKENS, MAX_TOKENS, OVERLAP_PCT)
    if not chunks:
        return None

    chunk_inputs = [E5_PREFIX + c for c in chunks]
    chunk_emb = _embed_texts(chunk_inputs, model, tokenizer, device)

    weights = [len(tokenizer.encode(E5_PREFIX + c, add_special_tokens=False)) for c in chunks]
    w = np.array(weights, dtype=np.float32)
    w = w / max(w.sum(), 1.0)

    doc_emb = (chunk_emb * w.reshape(-1, 1)).sum(axis=0)
    norm = np.linalg.norm(doc_emb)
    if norm > 0:
        doc_emb = doc_emb / norm

    return doc_emb.astype(np.float32).tolist()

# -----------------------------
# MAP IN PANDAS
# -----------------------------
out_schema = T.StructType([
    T.StructField(KEY_COL, key_dtype, nullable=False),
    T.StructField(EMBED_COL, T.ArrayType(T.FloatType()), nullable=True),
    T.StructField("embed_error", T.StringType(), nullable=True),
])

def embed_map(pdf_iter):

    for pdf in pdf_iter:
        keys = pdf[KEY_COL].tolist()
        texts = pdf[TEXT_COL].fillna("").astype(str).tolist()

        embs = []
        errs = []
        for t in texts:
            try:
                v = _embed_document(t)
                embs.append(v)
                errs.append(None if v is not None else "embed_document_returned_None")
            except Exception as e:
                embs.append(None)
                errs.append(str(e)[:2000])

        yield pd.DataFrame({KEY_COL: keys, EMBED_COL: embs, "embed_error": errs})

# -----------------------------
# STAGING TABLE
# -----------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{STAGING_TABLE}` (
  `{KEY_COL}` {key_dtype.simpleString()},
  `{EMBED_COL}` ARRAY<FLOAT>,
  `embed_error` STRING,
  `batch_id` BIGINT,
  `ingested_at` TIMESTAMP
)
USING DELTA
""")

log(f"Staging table ready: {STAGING_TABLE}")

# -----------------------------
# BATCH SELECTOR
# -----------------------------
def next_batch_df(limit_rows: int):
    return (
        spark.table(f"`{OCR_TABLE}`")
             .where(F.col(TEXT_COL).isNotNull() & (F.length(F.trim(F.col(TEXT_COL))) > 0))
             .where(F.col(EMBED_COL).isNull())
             .select(F.col(KEY_COL), F.col(TEXT_COL))
             .orderBy(F.col(KEY_COL))
             .limit(limit_rows)
    )

def has_rows(df):
    return len(df.take(1)) > 0

# -----------------------------
# ORCHESTRATOR
# -----------------------------
batch_id = 0
progress("initial progress")

while True:
    batch_df = next_batch_df(BATCH_ROWS)

    if not has_rows(batch_df):
        log("✅ Done: no rows left with embedding IS NULL.")
        break

    batch_id += 1
    log("=" * 60)
    log(f"Starting batch_id={batch_id} (rows={BATCH_ROWS}, partitions={NUM_PARTITIONS}, batch_size={BATCH_SIZE})")
    log("=" * 60)

    attempt = 0
    while True:
        attempt += 1
        t_batch0 = time.time()

        try:
            emb_df = (
                batch_df
                .repartition(NUM_PARTITIONS, F.col(KEY_COL))
                .mapInPandas(embed_map, schema=out_schema)
                .withColumn("batch_id", F.lit(batch_id).cast("bigint"))
                .withColumn("ingested_at", F.current_timestamp())
            )

            (emb_df
             .write
             .format("delta")
             .mode("append")
             .saveAsTable(STAGING_TABLE))

            staged = (spark.table(STAGING_TABLE)
                          .where(F.col("batch_id") == batch_id)
                          .selectExpr(
                              "count(*) as staged_rows",
                              f"sum(case when {EMBED_COL} is not null then 1 else 0 end) as staged_with_emb",
                              "sum(case when embed_error is not null then 1 else 0 end) as staged_errors"
                          )
                     )
            log("✅ Batch appended to staging:")
            staged.show(truncate=False)

            log("Merging into main table...")
            spark.sql(f"""
            MERGE INTO `{OCR_TABLE}` AS t
            USING (
              SELECT `{KEY_COL}` as `{KEY_COL}`, `{EMBED_COL}` as `{EMBED_COL}`
              FROM `{STAGING_TABLE}`
              WHERE batch_id = {batch_id} AND `{EMBED_COL}` IS NOT NULL
            ) AS s
            ON t.`{KEY_COL}` = s.`{KEY_COL}`
            WHEN MATCHED AND t.`{EMBED_COL}` IS NULL
              THEN UPDATE SET t.`{EMBED_COL}` = s.`{EMBED_COL}`
            """)

            elapsed = time.time() - t_batch0
            log(f"✅ Batch {batch_id} MERGE complete in {round(elapsed, 2)}s")
            progress(f"after batch {batch_id}")
            break

        except Exception as e:
            log(f"❌ Batch {batch_id} attempt {attempt} failed: {str(e)[:2000]}")
            if attempt >= MAX_RETRIES:
                log("🛑 Stopping after max retries. Progress is محفوظ in the staging table.")
                raise
            sleep_s = SLEEP_BASE_SEC * attempt
            log(f"Retrying batch {batch_id} in {sleep_s} seconds ...")
            time.sleep(sleep_s)

log("✅ All batches finished.")


---------------------------------------------------------------------------
RuntimeError                              Traceback (most recent call last)
File <command-4587247194011791>, line 104
     96     snapshot_download(
     97         repo_id="intfloat/multilingual-e5-large",
     98         local_dir=DBFS_MODEL_DIR,
     99         local_dir_use_symlinks=False
    100     )
    102     log(f"Downloaded model to {DBFS_MODEL_DIR} in {round(time.time() - t0, 2)}s")
--> 104 ensure_model_on_dbfs()
    106 # -----------------------------
    107 # TEXT CLEANING + CHUNKING
    108 # -----------------------------
    109 _MODEL = None

File <command-4587247194011791>, line 96, in ensure_model_on_dbfs()
     91 except Exception as e:
     92     raise Exception(
     93         "huggingface_hub not installed. Install it with: %pip install huggingface_hub"
     94     ) from e
---> 96 snapshot_download(
     97     repo_id="intfloat/multilingual-e5-large",
     98     local_dir=DBFS_MODEL

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

import re
import os
import time
import shutil
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

# -----------------------------
# CONFIG
# -----------------------------
OCR_TABLE   = "noc_documents_multilingual"
TEXT_COL    = "final_text"
EMBED_COL   = "embedding"
KEY_COL     = None 

MIN_TOKENS  = 800
MAX_TOKENS  = 1200
OVERLAP_PCT = 0.15
E5_PREFIX   = "passage: "

BATCH_ROWS      = 500
NUM_PARTITIONS  = 4

MAX_LEN         = 512
BATCH_SIZE      = 8   

MAX_RETRIES     = 3
SLEEP_BASE_SEC  = 10

STAGING_TABLE   = f"{OCR_TABLE}__e5_staging"

# Model paths
DBFS_MODEL_DIR  = "dbfs:/FileStore/models/intfloat_multilingual_e5_large"  
LOCAL_MODEL_DIR = "/local_disk0/hf_models/intfloat_multilingual_e5_large"  
LOCAL_MODEL_DIR_DBFS_FUSE = "/dbfs/FileStore/models/intfloat_multilingual_e5_large"  

spark.conf.set("spark.sql.shuffle.partitions", str(max(8, NUM_PARTITIONS * 2)))

# -----------------------------
# LOG HELPERS
# -----------------------------
def log(msg: str):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

def progress(label="progress"):
    df = spark.table(f"`{OCR_TABLE}`").selectExpr(
        "count(*) as total_rows",
        f"sum(case when {EMBED_COL} is not null then 1 else 0 end) as completed_embeddings",
        f"sum(case when {EMBED_COL} is null then 1 else 0 end) as remaining_embeddings",
        f"round(100.0 * sum(case when {EMBED_COL} is not null then 1 else 0 end)/count(*), 4) as pct_complete"
    )
    log(label)
    df.show(truncate=False)

# -----------------------------
# LOAD TABLE + KEY
# -----------------------------
base_df = spark.table(f"`{OCR_TABLE}`")

if KEY_COL is None:
    cols = [c.lower() for c in base_df.columns]
    if "path" in cols:
        KEY_COL = base_df.columns[cols.index("path")]
    elif "id" in cols:
        KEY_COL = base_df.columns[cols.index("id")]
    else:
        raise Exception("Set KEY_COL to a unique identifier column (path or id).")

log(f"Using KEY_COL = {KEY_COL}")
key_dtype = base_df.schema[KEY_COL].dataType

# -----------------------------
# ENSURE MODEL EXISTS 
# -----------------------------
def ensure_model_available():

    os.environ["HF_HUB_DISABLE_XET"] = "1"


    try:
        existing = dbutils.fs.ls(DBFS_MODEL_DIR)
        if existing and len(existing) > 0:
            log(f"Model already present on DBFS: {DBFS_MODEL_DIR}")
            return
    except Exception:
        pass

    log("Model not found on DBFS. Downloading to local disk (/local_disk0) ...")
    t0 = time.time()

    if os.path.exists(LOCAL_MODEL_DIR):
        shutil.rmtree(LOCAL_MODEL_DIR, ignore_errors=True)
    os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)

    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id="intfloat/multilingual-e5-large",
        local_dir=LOCAL_MODEL_DIR
    )

    log(f"Downloaded to local disk in {round(time.time() - t0, 2)}s")

    log(f"Copying model to DBFS: {DBFS_MODEL_DIR}")
    dbutils.fs.rm(DBFS_MODEL_DIR, recurse=True)
    dbutils.fs.mkdirs(DBFS_MODEL_DIR)

    t1 = time.time()
    src_root = LOCAL_MODEL_DIR
    dst_root = LOCAL_MODEL_DIR_DBFS_FUSE  # /dbfs/...
    os.makedirs(dst_root, exist_ok=True)

    for root, dirs, files in os.walk(src_root):
        rel = os.path.relpath(root, src_root)
        target_dir = os.path.join(dst_root, rel) if rel != "." else dst_root
        os.makedirs(target_dir, exist_ok=True)
        for fn in files:
            s = os.path.join(root, fn)
            d = os.path.join(target_dir, fn)
            shutil.copy2(s, d)

    log(f"Copied to DBFS in {round(time.time() - t1, 2)}s")
    log("✅ Model is now available for executors at: " + LOCAL_MODEL_DIR_DBFS_FUSE)

ensure_model_available()

# -----------------------------
# TEXT CLEANING + CHUNKING
# -----------------------------
_MODEL = None
_TOKENIZER = None
_DEVICE = None

_CTRL_CHARS_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")
_WS_RE = re.compile(r"\s+")
_SENT_SPLIT_RE = re.compile(r"(?<=[\.\!\?\;\:])\s+|\n+")

def _get_model():
    """Executor-safe: load from /dbfs local path only."""
    global _MODEL, _TOKENIZER, _DEVICE
    if _MODEL is None or _TOKENIZER is None:
        model_path = LOCAL_MODEL_DIR_DBFS_FUSE
        _TOKENIZER = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
        _MODEL = AutoModel.from_pretrained(model_path, local_files_only=True)
        _MODEL.eval()
        _DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        _MODEL.to(_DEVICE)
    return _MODEL, _TOKENIZER, _DEVICE

def _clean_text(s: str) -> str:
    if s is None:
        return ""
    s = _CTRL_CHARS_RE.sub(" ", s)
    s = _WS_RE.sub(" ", s).strip()
    return s

def _sentences(s: str):
    parts = _SENT_SPLIT_RE.split(s)
    return [p.strip() for p in parts if p and p.strip()]

def _build_chunks(sent_list, tokenizer, min_tokens, max_tokens, overlap_pct):
    if not sent_list:
        return []
    overlap_tokens = max(1, int(max_tokens * overlap_pct))
    chunks, cur_sents, cur_tok = [], [], 0

    sent_tok_lens = [len(tokenizer.encode(E5_PREFIX + t, add_special_tokens=False)) for t in sent_list]

    i = 0
    while i < len(sent_list):
        s = sent_list[i]
        s_tok = sent_tok_lens[i]

        if s_tok > max_tokens:
            if cur_sents:
                chunks.append(" ".join(cur_sents))
                cur_sents, cur_tok = [], 0

            ids = tokenizer.encode(E5_PREFIX + s, add_special_tokens=False)
            start = 0
            while start < len(ids):
                end = min(start + max_tokens, len(ids))
                sub = tokenizer.decode(ids[start:end])
                chunks.append(sub)
                if end == len(ids):
                    break
                start = max(0, end - overlap_tokens)
            i += 1
            continue

        if cur_tok + s_tok <= max_tokens:
            cur_sents.append(s)
            cur_tok += s_tok
            i += 1
        else:
            if cur_sents:
                chunks.append(" ".join(cur_sents))
            else:
                chunks.append(s)
                i += 1

            overlap_sents, overlap_count = [], 0
            j = len(cur_sents) - 1
            while j >= 0 and overlap_count < overlap_tokens:
                overlap_sents.append(cur_sents[j])
                overlap_count += len(tokenizer.encode(E5_PREFIX + cur_sents[j], add_special_tokens=False))
                j -= 1
            overlap_sents.reverse()

            cur_sents = overlap_sents
            cur_tok = 0
            for osent in cur_sents:
                cur_tok += len(tokenizer.encode(E5_PREFIX + osent, add_special_tokens=False))

    if cur_sents:
        chunks.append(" ".join(cur_sents))
    return chunks

# -----------------------------
# EMBEDDING
# -----------------------------
@torch.no_grad()
def _embed_texts(texts, model, tokenizer, device):
    all_emb = []
    for b0 in range(0, len(texts), BATCH_SIZE):
        batch = texts[b0:b0+BATCH_SIZE]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        out = model(**enc)
        last = out.last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1)

        summed = (last * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1)
        emb = summed / counts
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)

        all_emb.append(emb.detach().cpu().numpy())
        del enc, out, last, mask, summed, counts, emb

    return np.vstack(all_emb)

def _embed_document(doc_text: str):
    model, tokenizer, device = _get_model()
    cleaned = _clean_text(doc_text)
    if not cleaned:
        return None

    sents = _sentences(cleaned)
    chunks = _build_chunks(sents, tokenizer, MIN_TOKENS, MAX_TOKENS, OVERLAP_PCT)
    if not chunks:
        return None

    chunk_inputs = [E5_PREFIX + c for c in chunks]
    chunk_emb = _embed_texts(chunk_inputs, model, tokenizer, device)

    weights = [len(tokenizer.encode(E5_PREFIX + c, add_special_tokens=False)) for c in chunks]
    w = np.array(weights, dtype=np.float32)
    w = w / max(w.sum(), 1.0)

    doc_emb = (chunk_emb * w.reshape(-1, 1)).sum(axis=0)
    norm = np.linalg.norm(doc_emb)
    if norm > 0:
        doc_emb = doc_emb / norm

    return doc_emb.astype(np.float32).tolist()

# -----------------------------
# MAP IN PANDAS
# -----------------------------
out_schema = T.StructType([
    T.StructField(KEY_COL, key_dtype, nullable=False),
    T.StructField(EMBED_COL, T.ArrayType(T.FloatType()), nullable=True),
    T.StructField("embed_error", T.StringType(), nullable=True),
])

def embed_map(pdf_iter):
    for pdf in pdf_iter:
        keys = pdf[KEY_COL].tolist()
        texts = pdf[TEXT_COL].fillna("").astype(str).tolist()

        embs = []
        errs = []
        for t in texts:
            try:
                v = _embed_document(t)
                embs.append(v)
                errs.append(None if v is not None else "embed_document_returned_None")
            except Exception as e:
                embs.append(None)
                errs.append(str(e)[:2000])

        yield pd.DataFrame({KEY_COL: keys, EMBED_COL: embs, "embed_error": errs})

# -----------------------------
# STAGING TABLE
# -----------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{STAGING_TABLE}` (
  `{KEY_COL}` {key_dtype.simpleString()},
  `{EMBED_COL}` ARRAY<FLOAT>,
  `embed_error` STRING,
  `batch_id` BIGINT,
  `ingested_at` TIMESTAMP
)
USING DELTA
""")
log(f"Staging table ready: {STAGING_TABLE}")

# -----------------------------
# BATCH SELECTOR
# -----------------------------
def next_batch_df(limit_rows: int):
    return (
        spark.table(f"`{OCR_TABLE}`")
             .where(F.col(TEXT_COL).isNotNull() & (F.length(F.trim(F.col(TEXT_COL))) > 0))
             .where(F.col(EMBED_COL).isNull())
             .select(F.col(KEY_COL), F.col(TEXT_COL))
             .orderBy(F.col(KEY_COL))
             .limit(limit_rows)
    )

def has_rows(df):
    return len(df.take(1)) > 0

# -----------------------------
# ORCHESTRATOR
# -----------------------------
batch_id = 0
progress("Initial progress")

while True:
    batch_df = next_batch_df(BATCH_ROWS)

    if not has_rows(batch_df):
        log("✅ Done: no rows left with embedding IS NULL.")
        break

    batch_id += 1
    log("=" * 70)
    log(f"Starting batch_id={batch_id} | rows={BATCH_ROWS} | partitions={NUM_PARTITIONS} | batch_size={BATCH_SIZE}")
    log("=" * 70)

    attempt = 0
    while True:
        attempt += 1
        t0 = time.time()

        try:
            emb_df = (
                batch_df
                .repartition(NUM_PARTITIONS, F.col(KEY_COL))
                .mapInPandas(embed_map, schema=out_schema)
                .withColumn("batch_id", F.lit(batch_id).cast("bigint"))
                .withColumn("ingested_at", F.current_timestamp())
            )

            log("Writing embeddings to staging table ...")
            (emb_df.write.format("delta").mode("append").saveAsTable(STAGING_TABLE))

            staged_stats = (spark.table(STAGING_TABLE)
                .where(F.col("batch_id") == batch_id)
                .selectExpr(
                    "count(*) as staged_rows",
                    f"sum(case when {EMBED_COL} is not null then 1 else 0 end) as staged_with_emb",
                    "sum(case when embed_error is not null then 1 else 0 end) as staged_errors"
                )
            )

            log("✅ Batch appended to staging:")
            staged_stats.show(truncate=False)

            log("Merging into main table ...")
            spark.sql(f"""
            MERGE INTO `{OCR_TABLE}` AS t
            USING (
              SELECT `{KEY_COL}` as `{KEY_COL}`, `{EMBED_COL}` as `{EMBED_COL}`
              FROM `{STAGING_TABLE}`
              WHERE batch_id = {batch_id} AND `{EMBED_COL}` IS NOT NULL
            ) AS s
            ON t.`{KEY_COL}` = s.`{KEY_COL}`
            WHEN MATCHED AND t.`{EMBED_COL}` IS NULL
              THEN UPDATE SET t.`{EMBED_COL}` = s.`{EMBED_COL}`
            """)

            log(f"✅ Batch {batch_id} merge complete in {round(time.time() - t0, 2)}s")
            progress(f"After batch {batch_id}")
            break

        except Exception as e:
            log(f"❌ Batch {batch_id} attempt {attempt} failed: {str(e)[:2000]}")
            if attempt >= MAX_RETRIES:
                log("🛑 Stopping after max retries. Progress is محفوظ in the staging table.")
                raise
            sleep_s = SLEEP_BASE_SEC * attempt
            log(f"Retrying in {sleep_s}s ...")
            time.sleep(sleep_s)

log("✅ All batches finished.")


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can